# EV Fleet Activity Lakehouse

**Question:** how does observed vehicle movement vary by depot, model and wet/dry days?

**The map:** telemetry JSONL → Bronze → usable readings and quarantine → daily distance → fleet SQL database → weather API → Gold report.

Fleet and telemetry are synthetic. Weather is real Open-Meteo data. Distance covers the first to last reading **within a UTC day**, not all travel during that day.

**How to learn:** run setup, then rebuild one numbered step at a time. Each step shows something you can check. Start with Bronze; you do not need to understand the generator, test runner or report formatting first. The appendix saves evidence and formats the report. See `REBUILD.md` for small practice tasks.

**Databricks replay:** import `00_prepare_demo` into the same folder. The next cell prepares the source data and helper functions. No live cloud API fetch is claimed. The large setup code and saved JSON live in that separate notebook, so start your reading here.

### Prepare the inputs — run, then continue
`%run` executes the companion setup notebook and makes its paths/functions available here. It contains source generation and weather support only; the pipeline steps below stay visible.

In [ ]:
%run ./00_prepare_demo

## 0. Setup — run this, then start at Bronze
Imports and reporting dates. Databricks supplies Spark; the preparation notebook supplies input paths. Reruns overwrite only this demo’s outputs. The notebook is not one transaction.

In [ ]:
from pathlib import Path
from datetime import date, datetime, timezone
import json
import os
import sqlite3
import uuid

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

project_dir = Path(os.environ.get("FLEET_PROJECT_DIR", Path.cwd())).resolve()
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)
run_id = str(uuid.uuid4())

# This notebook uses the Databricks-provided Spark session.
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Report period:", start_date, "to", end_date)

## 1. Read telemetry into Bronze
Read the timestamp as text so a bad timestamp can be inspected later. PERMISSIVE keeps broken JSON in `_corrupt_record`. Save before querying that column. Expected: 4,316 rows.

In [ ]:
telemetry_schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True),
])
raw_telemetry = (spark.read.schema(telemetry_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(str(source_dir / "telemetry" / "*.jsonl"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id)))
bronze_table = schema_name + ".bronze_telemetry"
raw_telemetry.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)
bronze = spark.table(bronze_table)
bronze.printSchema()
print("Bronze rows:", bronze.count())

## 2. Clean IDs and read timestamps
`try_to_timestamp` returns null when text cannot be parsed. UTC is set in setup; `event_date` is the UTC day.

In [ ]:
parsed = (bronze
    .withColumn("vehicle_id", F.upper(F.trim("vehicle_id")))
    .withColumn("event_time", F.try_to_timestamp("event_timestamp"))
    .withColumn("event_date", F.to_date("event_time")))
parsed.select("vehicle_id", "event_timestamp", "event_time").show(5, truncate=False)

## 3. Label unusable readings
Read the checks from top to bottom: **the first failed check wins**. A null reason means usable so far. NaN and infinity are numeric special values, not distances. Expected: four rejected records.

In [ ]:
invalid_odometer = (
    F.col("odometer_km").isNull()
    | F.isnan("odometer_km")
    | (F.abs(F.col("odometer_km")) == float("inf"))
    | (F.col("odometer_km") < 0)
)
checked = parsed.withColumn("reject_reason",
    F.when(F.col("_corrupt_record").isNotNull(), "malformed_json")
     .when(F.col("vehicle_id").isNull() | (F.col("vehicle_id") == ""), "missing_vehicle")
     .when(F.col("event_time").isNull(), "invalid_timestamp")
     .when(~F.col("event_date").between(start_date, end_date), "outside_report_period")
     .when(invalid_odometer, "invalid_odometer")
     .otherwise(None))
base_rejects = checked.filter(F.col("reject_reason").isNotNull())
valid = checked.filter(F.col("reject_reason").isNull())
base_rejects.groupBy("reject_reason").count().show()

## 4. Remove identical readings
Compare only the business fields. Two copies of the same reading should count once. Expected: one duplicate removed.

In [ ]:
business_columns = ["vehicle_id", "event_time", "event_date", "odometer_km"]
unique_readings = valid.select(*business_columns).dropDuplicates()
exact_duplicates_removed = valid.count() - unique_readings.count()
print("Exact duplicates removed:", exact_duplicates_removed)

## 5. Find conflicting timestamps
After exact duplicates are removed, two readings at the same vehicle/time disagree. `left_anti` keeps rows with no match: leave these conflicts out before ordering readings.

In [ ]:
conflicting_keys = (unique_readings.groupBy("vehicle_id", "event_time")
    .count().filter(F.col("count") > 1).drop("count"))
conflicts = unique_readings.join(conflicting_keys, ["vehicle_id", "event_time"], "inner")
ordered_readings = unique_readings.join(conflicting_keys, ["vehicle_id", "event_time"], "left_anti")
conflicts.orderBy("vehicle_id", "event_time").show(truncate=False)

## 6. Compare each reading with the previous one
Use SQL `LAG`, partitioned by vehicle and UTC day. A falling pair tells us the day is inconsistent; it does not prove which reading is wrong.

In [ ]:
ordered_readings.createOrReplaceTempView("ordered_readings")
sequenced = spark.sql("""
    SELECT vehicle_id, event_time, event_date, odometer_km,
           LAG(odometer_km) OVER (
               PARTITION BY vehicle_id, event_date ORDER BY event_time
           ) AS previous_odometer
    FROM ordered_readings
""")
sequenced = sequenced.withColumn("delta_km", F.col("odometer_km") - F.col("previous_odometer"))
sequenced.filter(F.col("delta_km") < 0).show(truncate=False)

## 7. List days we cannot trust
A day is untrusted if it has a decrease, a conflict, or a rejected record we can assign to that day. `collect_set` keeps distinct reasons when several problems affect one day. `sort_array` only makes their list order repeatable. Expected: three days.

In [ ]:
key_columns = ["vehicle_id", "event_date"]
# A decrease identifies an inconsistent DAY, not which individual reading is wrong.
decrease_days = (sequenced.filter(F.col("delta_km") < 0).select(*key_columns)
    .withColumn("issue", F.lit("odometer_decreased")))
conflict_days = conflicts.select(*key_columns).withColumn("issue", F.lit("conflicting_timestamp"))
rejected_days = (base_rejects
    .filter(F.col("vehicle_id").isNotNull() & (F.col("vehicle_id") != "") &
            F.col("event_date").between(start_date, end_date))
    .select(*key_columns).withColumn("issue", F.lit("rejected_reading_in_day")))
day_issues = (decrease_days.unionByName(conflict_days).unionByName(rejected_days)
    .groupBy(*key_columns).agg(F.sort_array(F.collect_set("issue")).alias("reasons")))
issue_days = day_issues.select(*key_columns).withColumn("has_day_issue", F.lit(True))

day_issues.show(truncate=False)

## 8. Keep accepted Silver readings
Exclude every reading on a problem day using the day keys. This avoids keeping a spike just because the next reading went down.

In [ ]:
accepted = sequenced.join(issue_days.select(*key_columns), key_columns, "left_anti")
print("Accepted Silver readings:", accepted.count())

## 9. Keep rejected and held records
This inspection table uses a reasons list for both types of record. A rejected record gets its first failure; a held record gets the day’s reasons. `unionByName` lines up the columns; missing audit fields on held rows become null. Expected: four rejected + eighteen held.

In [ ]:
# Hold all usable-looking readings from an untrusted day; do not accuse one row.
held_readings = unique_readings.join(day_issues, key_columns, "inner")
quarantine_columns = business_columns + ["reasons"]
row_quarantine = (base_rejects.withColumn("reasons", F.array("reject_reason"))
    .select(*quarantine_columns, "event_timestamp", "_corrupt_record", "source_file")
    .withColumn("quarantine_scope", F.lit("record")))
day_quarantine = held_readings.select(*quarantine_columns).withColumn("quarantine_scope", F.lit("day_hold"))
quarantine = row_quarantine.unionByName(day_quarantine, allowMissingColumns=True)
quarantine.groupBy("quarantine_scope").count().show()

## 10. Calculate daily distance
One row per vehicle/day. Calculate max minus min only when the day passed the checks and has at least two readings. Two equal readings give zero; missing or unreliable readings give null.

In [ ]:
daily_observations = (sequenced.groupBy(*key_columns).agg(
    F.count("*").alias("reading_count"),
    F.min("event_time").alias("first_reading"),
    F.max("event_time").alias("last_reading"),
    (F.max("odometer_km") - F.min("odometer_km")).alias("observed_change_km")))
# A full join retains days whose only readings were rejected.
daily = (daily_observations.join(issue_days, key_columns, "full")
    .withColumn("reading_count", F.coalesce("reading_count", F.lit(0))))
daily = daily.withColumn("distance_status",
    F.when(F.col("has_day_issue"), "untrusted")
     .when(F.col("reading_count") < 2, "insufficient_readings")
     .otherwise("trusted"))
daily = (daily.withColumn("observed_distance_km",
    F.when(F.col("distance_status") == "trusted", F.round("observed_change_km", 2)))
    .drop("observed_change_km", "has_day_issue")
    .filter(F.col("event_date").between(start_date, end_date)))

daily.groupBy("distance_status").count().show()

## 11. Save the telemetry results
Save the day issues, accepted readings, quarantine and daily observations as separate Delta tables.

In [ ]:
day_issues.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".silver_day_issues")
accepted.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".silver_telemetry")
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".quarantine_telemetry")
daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".daily_observations")
daily = spark.table(schema_name + ".daily_observations")
daily.orderBy("event_date", "vehicle_id").show(8, truncate=False)

## 12. Query the fleet database
SQLite represents a separate fleet system. SQL supplies each vehicle’s model and depot. The depot coordinates will tell the weather API which locations to request.

In [ ]:
database_path = source_dir / "fleet.sqlite"
# Query the driver copy; database_path above is the identical volume copy used for input hashes.
with sqlite3.connect((local_source_dir / "fleet.sqlite").as_uri() + "?mode=ro", uri=True) as database:
    fleet_rows = database.execute("""
        SELECT UPPER(TRIM(v.vehicle_id)), v.model, UPPER(TRIM(v.depot_id)),
               d.depot_name, d.latitude, d.longitude
        FROM vehicles AS v
        LEFT JOIN depots AS d ON v.depot_id = d.depot_id
        ORDER BY v.vehicle_id
    """).fetchall()
    depot_rows = database.execute("SELECT depot_id, latitude, longitude FROM depots ORDER BY depot_id").fetchall()
depots = []
for row in depot_rows:
    depot = {"depot_id": row[0], "latitude": row[1], "longitude": row[2]}
    depots.append(depot)
fleet = spark.createDataFrame(fleet_rows,
    "vehicle_id string, model string, depot_id string, depot_name string, latitude double, longitude double")

fleet.show(4, truncate=False)

## 13. Check and save the roster
An empty roster, duplicated vehicle ID or incomplete depot could hide activity or multiply a join. Stop here if the roster is incomplete.

In [ ]:
if fleet.count() == 0:
    raise ValueError("Fleet database is empty")
if fleet.select("vehicle_id").distinct().count() != fleet.count():
    raise ValueError("Duplicate fleet vehicle ID")
incomplete_fleet = fleet.filter(
    F.col("vehicle_id").isNull() | (F.col("vehicle_id") == "") |
    F.col("model").isNull() | F.col("depot_id").isNull() |
    F.col("depot_name").isNull() | F.col("latitude").isNull() |
    F.col("longitude").isNull())
if incomplete_fleet.count() != 0:
    raise ValueError("Incomplete fleet record")
fleet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".bronze_fleet")
fleet = spark.table(schema_name + ".bronze_fleet")
print("Fleet rows checked and saved:", fleet.count())

## 14. Build the reporting dates
One date per day in the reporting period. A simple loop builds the small demo calendar.

In [ ]:
from datetime import timedelta

calendar_rows = []
current_day = start_date
while current_day <= end_date:
    # The trailing comma makes a one-column row.
    calendar_rows.append((current_day,))
    current_day = current_day + timedelta(days=1)
report_dates = spark.createDataFrame(calendar_rows, "event_date date")
print("Reporting days:", report_dates.count())

## 15. Keep missing and unknown vehicle-days
`crossJoin` creates the expected vehicle × day combinations. A left join keeps missing telemetry. The anti-join finds vehicles absent from the roster. Expected: 720 roster-days + one unknown vehicle-day.

In [ ]:
expected_vehicle_days = fleet.select("vehicle_id").crossJoin(report_dates)
registered_days = expected_vehicle_days.join(daily, ["vehicle_id", "event_date"], "left")
unregistered_days = daily.join(fleet.select("vehicle_id"), "vehicle_id", "left_anti")
vehicle_days = registered_days.unionByName(unregistered_days)
vehicle_days = (vehicle_days.join(fleet, "vehicle_id", "left")
    .withColumn("registered_vehicle", F.col("depot_id").isNotNull()))
vehicle_days = (vehicle_days
    .withColumn("distance_status", F.coalesce("distance_status", F.lit("missing_telemetry")))
    .withColumn("reading_count", F.coalesce("reading_count", F.lit(0)))
    .withColumn("depot_id", F.coalesce("depot_id", F.lit("UNASSIGNED")))
    .withColumn("depot_name", F.coalesce("depot_name", F.lit("Unknown depot")))
    .withColumn("model", F.coalesce("model", F.lit("Unknown model"))))
vehicle_days.groupBy("distance_status").count().show()

## 16. Retrieve or reuse weather
The weather helper in `00_prepare_demo` (also in `weather.py` in GitHub) handles requests, saved snapshots and parsing. Learn `parse_daily` first: it reads one index from each daily array and appends a row. Retry/cache handling is a later troubleshooting topic. Offline mode explicitly reads saved real responses; an API failure is not null weather.

In [ ]:
# load_weather is defined by 00_prepare_demo.

offline = os.environ.get("FLEET_OFFLINE") == "1"
weather_dir = project_dir / ("examples/weather" if offline else "data/weather")
weather_rows, weather_snapshots = load_weather(
    depots, start_date, end_date, weather_dir,
    refresh=os.environ.get("FLEET_REFRESH_WEATHER") == "1", offline=offline)
weather = spark.createDataFrame(weather_rows,
    "depot_id string, event_date date, temperature_c double, precipitation_mm double")
expected_weather_rows = len(depots) * ((end_date - start_date).days + 1)
if weather.count() != expected_weather_rows:
    raise ValueError(f"Expected {expected_weather_rows} depot-days of weather")
if weather.select("depot_id", "event_date").distinct().count() != weather.count():
    raise ValueError("Duplicate weather depot/date key")
weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".silver_weather")
weather = spark.table(schema_name + ".silver_weather")
weather.show(4, truncate=False)

## 17. Join weather and publish Gold
Join on **depot and date**. A weather row must match many vehicles without multiplying their rows. Precipitation ≥ 1 mm is Wet; a successful null observation is Unknown.

In [ ]:
gold = (vehicle_days.join(weather, ["depot_id", "event_date"], "left")
    .withColumn("weather_condition", F.when(F.col("precipitation_mm").isNull(), "Unknown")
        .when(F.col("precipitation_mm") >= 1.0, "Wet").otherwise("Dry"))
    .select("event_date", "vehicle_id", "depot_id", "depot_name", "model", "registered_vehicle",
        "reading_count", "first_reading", "last_reading", "distance_status", "observed_distance_km",
        "temperature_c", "precipitation_mm", "weather_condition"))
if gold.count() != vehicle_days.count():
    raise ValueError("Weather join multiplied or dropped vehicle-days")
if gold.select("event_date", "vehicle_id").distinct().count() != gold.count():
    raise ValueError("Gold grain is not unique")
non_trusted_distance = gold.filter(
    (F.col("distance_status") != "trusted") & F.col("observed_distance_km").isNotNull())
if non_trusted_distance.count() != 0:
    raise ValueError("Non-trusted day has a distance")
if gold.filter(F.col("observed_distance_km") < 0).count() != 0:
    raise ValueError("Negative observed distance")

gold_table = schema_name + ".gold_daily_vehicle_activity"
gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)
gold = spark.table(gold_table)
gold.createOrReplaceTempView("daily_vehicle_activity")

## 18. Answer the question with SQL
Show coverage beside averages so an unreliable or missing day is not mistaken for zero activity.

In [ ]:
activity_report = spark.sql("""
    SELECT depot_name, model, weather_condition,
           COUNT(*) AS vehicle_days,
           COUNT(observed_distance_km) AS trusted_vehicle_days,
           SUM(CASE WHEN distance_status = 'missing_telemetry' THEN 1 ELSE 0 END) AS missing_days,
           SUM(CASE WHEN distance_status = 'untrusted' THEN 1 ELSE 0 END) AS untrusted_days,
           SUM(CASE WHEN distance_status = 'insufficient_readings' THEN 1 ELSE 0 END) AS insufficient_days,
           ROUND(AVG(observed_distance_km), 2) AS mean_observed_distance_km,
           ROUND(SUM(observed_distance_km), 2) AS total_observed_distance_km
    FROM daily_vehicle_activity
    GROUP BY depot_name, model, weather_condition
    ORDER BY depot_name, model, weather_condition
""")
activity_report.show(truncate=False)

## Stop here for your first learning pass
You have followed all three sources to the report. Use `REBUILD.md` to recreate one step in a blank notebook and explain its output.

## Appendix — exports and presentation
These cells save CSVs, record run evidence and format HTML. Run them for a complete demo, but they are not prerequisites for learning Bronze, Silver and Gold.

In [ ]:
# These are small demo outputs; collecting is deliberate, not a large-data pattern.
import csv
# Format timestamps in Spark's UTC session before Python collects them.
daily_export = (gold.orderBy("event_date", "vehicle_id")
    .withColumn("first_reading", F.date_format("first_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("last_reading", F.date_format("last_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'")))
for name, frame in [("daily_vehicle_activity", daily_export),
                    ("activity_report", activity_report)]:
    with (output_dir / f"{name}.csv").open("w", newline="") as file:
        writer = csv.writer(file, lineterminator="\n")
        writer.writerow(frame.columns)
        writer.writerows(tuple(row) for row in frame.collect())

### Record what actually ran

In [ ]:
import hashlib
unassigned_rejects = quarantine.filter(
    F.col("vehicle_id").isNull() | (F.col("vehicle_id") == "") | F.col("event_date").isNull()).count()
record_reason_counts = {row.reason: row["count"] for row in
    row_quarantine.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
day_issue_counts = {row.reason: row["count"] for row in
    day_issues.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
status_counts = {row.distance_status: row["count"] for row in gold.groupBy("distance_status").count().collect()}
input_files = sorted((source_dir / "telemetry").glob("*.jsonl")) + [database_path]
input_files += sorted(weather_dir.glob("*.json"))
summary = {
    "run_id": run_id, "completed_at": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version, "period": [str(start_date), str(end_date)],
    "bronze_rows": bronze.count(), "exact_duplicates_removed": exact_duplicates_removed,
    "quarantined_rows": quarantine.count(), "unassigned_rejects": unassigned_rejects,
    "individual_reject_rows": row_quarantine.count(), "held_rows": day_quarantine.count(),
    "individual_reject_reason_counts": record_reason_counts, "untrusted_day_reason_counts": day_issue_counts,
    "gold_rows": gold.count(), "status_counts": status_counts,
    "unregistered_vehicle_days": gold.filter(~F.col("registered_vehicle")).count(),
    "weather_rows": weather.count(), "weather_snapshots": weather_snapshots,
    "inputs": {str(path.relative_to(project_dir)): hashlib.sha256(path.read_bytes()).hexdigest() for path in input_files},
    "business_output_sha256": hashlib.sha256((output_dir / "daily_vehicle_activity.csv").read_bytes()).hexdigest(),
}
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps({key: value for key, value in summary.items() if key not in ["inputs", "weather_snapshots"]}, indent=2))

### Format the standalone report

In [ ]:
import html
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

chart_rows = activity_report.filter(F.col("mean_observed_distance_km").isNotNull()).collect()
labels = [f"{row.depot_name} / {row.model} / {row.weather_condition}" for row in chart_rows]
values = [row.mean_observed_distance_km for row in chart_rows]
fig, axis = plt.subplots(figsize=(10, 5))
axis.barh(labels, values, color="#267a79")
axis.set_xlabel("Mean observed distance per trusted vehicle-day (km)")
axis.set_title("Synthetic fleet activity by depot, model and weather")
axis.invert_yaxis()
fig.tight_layout()
fig.savefig(output_dir / "activity.png", dpi=160)
plt.close(fig)

headers = "".join(f"<th>{html.escape(column.replace('_', ' '))}</th>" for column in activity_report.columns)
body = "".join("<tr>" + "".join(f"<td>{html.escape(str(value)) if value is not None else 'Unknown'}</td>" for value in row) + "</tr>"
    for row in activity_report.collect())
report_html = f"""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>EV Fleet Activity</title><style>body{{font:16px system-ui;max-width:1150px;margin:40px auto;padding:0 20px;color:#18313b;background:#f5f8fa}}h1{{font-size:36px}}img{{max-width:100%}}table{{border-collapse:collapse;width:100%;font-size:13px}}th,td{{padding:10px;border-bottom:1px solid #ccd9df;text-align:left}}th{{background:#dcebe9}}.table{{overflow:auto}}.note{{background:white;padding:20px;border-radius:12px}}</style>
<h1>EV Fleet Activity</h1><p>{start_date}–{end_date} · UTC reporting days · {summary['gold_rows']} vehicle-days</p>
<div class="note"><b>What this report answers</b><p>How does observed vehicle movement vary across depots, models and wet/dry days?</p>
<p>Fleet and telemetry are synthetic. Weather is real historical reanalysis from <a href="https://open-meteo.com/en/docs/historical-weather-api">Open-Meteo</a> (CC BY 4.0). This demonstrates integration, not real-world weather effects. Depot weather does not describe each vehicle's route.</p>
<p>Distance covers the first to last reading within a UTC day. Days with questionable readings or fewer than two usable readings have no distance estimate. Missing telemetry is unknown activity.</p></div>
<p>Trusted: {status_counts.get('trusted', 0)} · Untrusted: {status_counts.get('untrusted', 0)} · Insufficient: {status_counts.get('insufficient_readings', 0)} · Missing: {status_counts.get('missing_telemetry', 0)} · Unregistered vehicle-days: {summary['unregistered_vehicle_days']} · Unassigned rejected rows: {unassigned_rejects}</p>
<img src="activity.png" alt="Mean observed distance for each depot, model and weather group">
<div class="table"><table><thead><tr>{headers}</tr></thead><tbody>{body}</tbody></table></div>
<p>Run {run_id} · {summary['completed_at']}</p></html>"""
(output_dir / "report.html").write_text(report_html)
print("Report saved:", output_dir / "report.html")

### Verify the cloud result against the local reference

In [ ]:
# Compare the platform result with the verified local business output.
expected_hash = '766345118556c0b8901c8fbed8998a143c7741910c9faac6daade370e532c44c'
if summary["business_output_sha256"] != expected_hash:
    raise ValueError("Databricks business CSV differs from the local reference")
if summary["gold_rows"] != 721 or summary["quarantined_rows"] != 22:
    raise ValueError("Databricks counts differ from the local reference")
print("PLATFORM_REPLAY_VALIDATION_PASSED")
print("Weather mode: saved real API snapshots; no cloud fetch claimed")
print("Gold table:", gold_table)
print("Business SHA-256:", summary["business_output_sha256"])
print("Gold versions:")
spark.sql(f"DESCRIBE HISTORY {gold_table}").select("version", "timestamp", "operation").show()
display(activity_report)